# 08 — Manuscript Tables

Notebooks 03–06 write *analysis* CSVs: raw numbers at full precision with
internal column names like `macro_F1_95CI` and `model_key`. Those are the
right thing for computation and the wrong thing to paste into a manuscript.

This notebook turns them into **camera-ready tables** — merged, rounded to a
consistent precision, with the column headers and row labels you actually
want in print, significance markers attached, and a caption written for each
one. Every table is emitted three ways:

| Format | Use |
|---|---|
| `.csv` | inspection, co-author review, spreadsheet edits |
| `.tex` | `\input{}` straight into a LaTeX manuscript (booktabs) |
| `.md` | Word/Google Docs — paste and convert to a table |

Output lands in `artifacts/tables/`, plus `manuscript_tables.zip`.

Table numbering follows the paper outline: **T1–T7** in the main text,
**S1–S9** supplementary. Any table whose inputs are missing is reported in
the completeness audit at the end rather than silently skipped, so you always
know exactly what is still outstanding.

In [1]:
import os
import sys
import json
import re
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

ART = Path("/kaggle/working/artifacts")
TBL = ART / "tables"
TBL.mkdir(parents=True, exist_ok=True)


def find_file(fname):
    for base in (ART, ART / "embeddings", ART / "tables"):
        if (base / fname).exists():
            return base / fname
    hits = sorted(Path("/kaggle/input").rglob(fname))
    return hits[0] if hits else None


def load_csv(f):
    p = find_file(f)
    return pd.read_csv(p) if p is not None else None


def load_json(f):
    p = find_file(f)
    return json.loads(p.read_text()) if p is not None else None


REGISTRY = []      # every table we successfully build
MISSING = []       # every table we could not build, and why

# Display names used consistently across every table.
PRETTY = {
    "prior": "Class-prior baseline", "majority": "Majority-label baseline",
    "tfidf_logreg": "TF-IDF + logistic regression", "tfidf_svm": "TF-IDF + linear SVM",
    "clinicalbert": "ClinicalBERT", "biobert": "BioBERT",
    "img_resnet50": "ResNet50 + attention MIL",
    "img_densenet121": "DenseNet121 + attention MIL",
    "img_effnetv2s": "EfficientNetV2-S + attention MIL",
    "fusion_late": "Late fusion (image + text)",
    "zeroshot": "BiomedCLIP zero-shot", "probe_image": "BiomedCLIP linear probe (image)",
    "probe_text_findings": "BiomedCLIP linear probe (findings)",
    "probe_concat": "BiomedCLIP linear probe (image + findings)",
    "multitask": "Multi-task VLM (ours)",
    "constant_normal": "Constant \"normal\" (degenerate)",
    "random_train_report": "Random training report",
    "retrieve_copy@1": "Retrieve-and-copy (K=1)",
    "consensus_fusion@K": "Consensus fusion (ours)",
    "section_aware_fusion": "Section-aware fusion (ours)",
    "copy_full_report@1": "Copy full report (K=1)",
    "multitask_decoder": "Multi-task decoder",
    "rag_llm_rewrite": "RAG + LLM rewriting",
}
nice = lambda s: PRETTY.get(str(s), str(s))

## Formatting helpers

Three conventions applied everywhere, so the tables look like they came from
one paper rather than seven notebooks:

- **3 decimal places** for all metrics. Reporting 4+ implies a precision that
  259 studies cannot support.
- **`0.719 [0.636, 0.771]`** for a point estimate with its interval, in a
  single column. Splitting a CI across three columns wastes width and reads
  badly in a two-column journal layout.
- **Bold the best row** per metric, excluding degenerate baselines — they
  stay in the table but never win it.

In [2]:
def fmt(x, d=3):
    if x is None or (isinstance(x, float) and not np.isfinite(x)):
        return "—"
    try:
        return f"{float(x):.{d}f}"
    except (TypeError, ValueError):
        return str(x)


def fmt_ci(point, lo, hi, d=3):
    if any(v is None or (isinstance(v, float) and not np.isfinite(v))
           for v in (point, lo, hi)):
        return fmt(point, d)
    return f"{float(point):.{d}f} [{float(lo):.{d}f}, {float(hi):.{d}f}]"


def parse_ci(s):
    """'[0.636, 0.771]' -> (0.636, 0.771)"""
    try:
        lo, hi = str(s).strip("[] ").split(",")
        return float(lo), float(hi)
    except Exception:
        return np.nan, np.nan


def fmt_p(p):
    try:
        p = float(p)
    except (TypeError, ValueError):
        return "—"
    if p < 1e-4:
        return "<0.0001"
    if p < 0.001:
        return f"{p:.2e}"
    return f"{p:.4f}"


def stars(p):
    try:
        p = float(p)
    except (TypeError, ValueError):
        return ""
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""


def bold_best(df, col, exclude_rows=(), higher_better=True):
    """Wrap the winning cell in ** ** (markdown) — converted to \\textbf in LaTeX."""
    vals = []
    for i, v in enumerate(df[col]):
        if df.iloc[i, 0] in exclude_rows:
            vals.append(np.nan)
            continue
        m = re.match(r"^(-?[\d.]+)", str(v))
        vals.append(float(m.group(1)) if m else np.nan)
    vals = np.array(vals, dtype=float)
    if np.all(np.isnan(vals)):
        return df
    k = int(np.nanargmax(vals) if higher_better else np.nanargmin(vals))
    out = df.copy()
    out.iloc[k, out.columns.get_loc(col)] = f"**{out.iloc[k][col]}**"
    return out


def to_latex(df, caption, label, note=""):
    body = df.copy()
    for c in body.columns:
        body[c] = body[c].astype(str).apply(
            lambda s: re.sub(r"\*\*(.+?)\*\*", r"\\textbf{\1}", s))
        body[c] = body[c].str.replace("%", r"\%", regex=False)
        body[c] = body[c].str.replace("_", r"\_", regex=False)
        body[c] = body[c].str.replace("—", "--", regex=False)
        body[c] = body[c].str.replace("±", r"$\pm$", regex=False)
        body[c] = body[c].str.replace("✓", r"$\checkmark$", regex=False)
        body[c] = body[c].str.replace("✗", r"$\times$", regex=False)
        body[c] = body[c].str.replace("×", r"$\times$", regex=False)
        body[c] = body[c].str.replace("Δ", r"$\Delta$", regex=False)
        body[c] = body[c].str.replace("κ", r"$\kappa$", regex=False)
    cols = "l" + "r" * (body.shape[1] - 1)

    def esc_head(c):
        # Headers need the same escaping as cells. An unescaped % here comments
        # out the rest of the LaTeX line and silently destroys the table.
        c = str(c)
        for a, b in (("\\", r"\textbackslash "), ("%", r"\%"), ("_", r"\_"),
                     ("&", r"\&"), ("#", r"\#"), ("✓", r"$\checkmark$"),
                     ("✗", r"$\times$"), ("Δ", r"$\Delta$"), ("—", "--")):
            c = c.replace(a, b)
        return c

    head = " & ".join(esc_head(c) for c in body.columns) + r" \\"
    rows = "\n".join(" & ".join(r) + r" \\" for r in body.astype(str).values)
    tex = [r"\begin{table}[htbp]", r"\centering",
           f"\\caption{{{caption}}}", f"\\label{{{label}}}",
           r"\small", f"\\begin{{tabular}}{{{cols}}}", r"\toprule",
           head, r"\midrule", rows, r"\bottomrule", r"\end{tabular}"]
    if note:
        tex.append(r"\begin{flushleft}\footnotesize " + note + r"\end{flushleft}")
    tex.append(r"\end{table}")
    return "\n".join(tex)


def emit(df, tid, title, caption, note="", show=True):
    """Write csv/tex/md for one table and register it."""
    if df is None or len(df) == 0:
        MISSING.append({"id": tid, "title": title, "reason": "no data"})
        print(f"[missing] {tid} — {title}")
        return None
    base = TBL / f"{tid}_{re.sub(r'[^a-z0-9]+', '_', title.lower()).strip('_')}"
    df.to_csv(base.with_suffix(".csv"), index=False)
    base.with_suffix(".tex").write_text(
        to_latex(df, caption, f"tab:{tid.lower()}", note))
    md = [f"**{tid}. {caption}**", "",
          "| " + " | ".join(map(str, df.columns)) + " |",
          "|" + "|".join(["---"] * df.shape[1]) + "|"]
    md += ["| " + " | ".join(map(str, r)) + " |" for r in df.astype(str).values]
    if note:
        md += ["", f"_{note}_"]
    base.with_suffix(".md").write_text("\n".join(md))
    REGISTRY.append({"id": tid, "title": title, "rows": len(df),
                     "cols": df.shape[1], "file": base.name, "caption": caption})
    print(f"[ok] {tid} — {title}  ({len(df)}x{df.shape[1]})")
    if show:
        print(df.to_string(index=False))
        print()
    return df

---
# Main text tables

## T1 — Dataset characteristics

The first table a reviewer reads. It must answer: how much data, of what,
how imbalanced, and how was it split — without them having to hunt.

In [3]:
df = load_csv("master_split.csv")
card = load_json("dataset_card.json") or {}
frames_df = load_csv("frames.csv")
tiers = load_json("label_tiers.json") or {}
Ydf = load_csv("labels_multihot.csv")

if df is not None:
    n = len(df)
    rows = [("Studies", f"{n}"),
            ("Ultrasound frames after decomposition",
             f"{len(frames_df)}" if frames_df is not None else "—"),
            ("Frames per study, mean (range)",
             (f"{len(frames_df)/n:.1f} "
              f"({frames_df.groupby('study_id').size().min()}–"
              f"{frames_df.groupby('study_id').size().max()})")
             if frames_df is not None else "—")]

    if "exam_type" in df.columns:
        vc = df["exam_type"].value_counts()
        rows.append(("Examination types", f"{len(vc)}"))
        for k, v in vc.head(6).items():
            rows.append((f"    {k}", f"{v} ({v/n:.0%})"))

    num = Ydf.select_dtypes("number") if Ydf is not None else None
    if num is not None:
        rows += [("Distinct findings in ontology", f"{num.shape[1]}"),
                 ("Findings modelled (trainable + marginal tiers)",
                  f"{len(tiers.get('trainable', [])) + len(tiers.get('marginal', []))}"),
                 ("Findings excluded as rare (n < 5)", f"{len(tiers.get('rare', []))}"),
                 ("Studies with >1 finding",
                  f"{int((num.sum(axis=1) > 1).sum())} "
                  f"({(num.sum(axis=1) > 1).mean():.0%})"),
                 ("Most prevalent finding",
                  f"{num.sum(0).idxmax()} ({int(num.sum(0).max())}, "
                  f"{num.sum(0).max()/n:.0%})")]

    for key, lab in [("mean_report_words", "Report length, mean words"),
                     ("mean_impression_words", "Impression length, mean words"),
                     ("mean_sections", "Sections per report, mean")]:
        if key in card:
            rows.append((lab, f"{card[key]:.1f}"))

    if "fold" in df.columns:
        sizes = df["fold"].value_counts().sort_index()
        rows.append(("Cross-validation",
                     f"{len(sizes)}-fold, group-stratified "
                     f"({sizes.min()}–{sizes.max()} studies per fold)"))
    if "dup_group" in df.columns:
        rows.append(("Near-duplicate groups kept within a fold",
                     f"{n - df['dup_group'].nunique()} studies in "
                     f"{df['dup_group'].nunique()} groups"))

    t1 = pd.DataFrame(rows, columns=["Characteristic", "Value"])
    emit(t1, "T1", "Dataset characteristics",
         "Characteristics of the ultrasound study corpus.",
         "Findings were extracted from the Impression section using the rule "
         "ontology in Table S1. Percentages are of all studies.")
else:
    MISSING.append({"id": "T1", "title": "Dataset characteristics",
                    "reason": "master_split.csv not found"})

[ok] T1 — Dataset characteristics  (20x2)
                                Characteristic                                             Value
                                       Studies                                               259
         Ultrasound frames after decomposition                                               494
                Frames per study, mean (range)                                        1.9 (1–12)
                             Examination types                                                 8
                                 Whole abdomen                                         116 (45%)
                                 Lower abdomen                                          56 (22%)
                                           TVS                                          32 (12%)
                             Pregnancy profile                                          32 (12%)
                                         Other                                       

## T2 — Implementation details

Assembled from the saved configs, with any value the notebooks did not record
left as `TODO` for you to fill. Reviewers check this table for
reproducibility; a blank cell here is a revision request.

In [4]:
pre = load_json("preprocess_config.json") or {}
vlm = load_json("vlm_config.json") or {}

rows = [
    ("Frame resolution", f"{pre.get('img_size', 224)} × {pre.get('img_size', 224)}"),
    ("Contrast enhancement", pre.get("clahe", "CLAHE, clip 2.0, 8×8 grid")),
    ("Speckle reduction", pre.get("denoise", "TODO")),
    ("Augmentation",
     "rotation ±10°, brightness/contrast jitter, gain noise, random crop 0.85–1.0"),
    ("Horizontal flip", "Not used — laterality is clinically meaningful"),
    ("Image backbones", "ResNet50, DenseNet121, EfficientNetV2-S (ImageNet init)"),
    ("Bag aggregation", "Gated attention MIL (Ilse et al., 2018)"),
    ("Text encoder", "ClinicalBERT (emilyalsentzer/Bio_ClinicalBERT)"),
    ("Vision–language backbone", vlm.get("backbone", "BiomedCLIP ViT-B/16")),
    ("Report decoder", vlm.get("decoder", "distilGPT-2, 8-token image prefix")),
    ("Generation target", vlm.get("generation_target", "impression")),
    ("Loss weights (cls / gen / contrastive)",
     " / ".join(str(vlm.get("lambda", {}).get(k, "—"))
                for k in ("cls", "gen", "con"))),
    ("Unfrozen ViT blocks", str(vlm.get("n_unfrozen_blocks", "—"))),
    ("Optimiser", "AdamW, weight decay 1e-2 (heads 1e-4)"),
    ("Learning rate (backbone / head)", "1e-4 / 1e-3 (CNN); 2e-5 / 1e-3 (VLM)"),
    ("Schedule", "Cosine annealing, no warm restarts"),
    ("Epochs", f"{vlm.get('epochs', 25)} (CNN 25, VLM {vlm.get('epochs', 20)})"),
    ("Batch size", "8 studies (variable-size frame bags, padded + masked)"),
    ("Class imbalance", "Positive-weighted BCE, weights clipped to [1, 15]"),
    ("Decision thresholds", "Per class, tuned on training folds only"),
    ("Retrieval index", "FAISS IndexFlatIP over L2-normalised embeddings"),
    ("Retrieval exclusion", "Query's own fold and duplicate group removed"),
    ("Random seed", "42 (numpy, torch, split generation)"),
    ("Hardware", "TODO — e.g. NVIDIA T4 16 GB, Kaggle"),
    ("Framework", "TODO — PyTorch x.y, timm x.y, transformers x.y"),
]
t2 = pd.DataFrame(rows, columns=["Setting", "Value"])
emit(t2, "T2", "Implementation details",
     "Implementation and training configuration.",
     "Values marked TODO must be completed from your run environment.",
     show=False)
todo = t2[t2["Value"].astype(str).str.contains("TODO")]
print(t2.to_string(index=False))
if len(todo):
    print(f"\n>>> {len(todo)} rows still need filling in: "
          f"{', '.join(todo['Setting'])}")

[ok] T2 — Implementation details  (25x2)
                               Setting                                                                       Value
                      Frame resolution                                                                   224 × 224
                  Contrast enhancement                                                   CLAHE, clip 2.0, 8×8 grid
                     Speckle reduction                                                                        TODO
                          Augmentation rotation ±10°, brightness/contrast jitter, gain noise, random crop 0.85–1.0
                       Horizontal flip                              Not used — laterality is clinically meaningful
                       Image backbones                     ResNet50, DenseNet121, EfficientNetV2-S (ImageNet init)
                       Bag aggregation                                     Gated attention MIL (Ilse et al., 2018)
                          Text encoder 

## T3 — Classification results

Point estimates with bootstrap intervals in one column each. Degenerate
baselines stay in the table and are excluded from bolding.

In [5]:
ci = load_csv("results_with_ci.csv")
perm = load_csv("permutation_tests.csv")

if ci is not None:
    rows = []
    for _, r in ci.iterrows():
        lo_mi, hi_mi = parse_ci(r.get("micro_F1_95CI"))
        lo_ma, hi_ma = parse_ci(r.get("macro_F1_95CI"))
        lo_au, hi_au = parse_ci(r.get("macro_AUC_95CI"))
        row = {"Model": nice(r["model"]),
               "Micro-F1 [95% CI]": fmt_ci(r.get("micro_F1"), lo_mi, hi_mi),
               "Macro-F1 [95% CI]": fmt_ci(r.get("macro_F1"), lo_ma, hi_ma),
               "Macro-AUC [95% CI]": fmt_ci(r.get("macro_AUC"), lo_au, hi_au)}
        if perm is not None:
            m = perm[perm["model"] == r["model"]]
            if len(m):
                p = m.iloc[0]["p_permutation"]
                row["p (perm.)"] = fmt_p(p) + stars(p)
        rows.append(row)

    t3 = pd.DataFrame(rows)
    t3["_sort"] = [float(str(v).split(" ")[0]) for v in t3["Macro-F1 [95% CI]"]]
    t3 = t3.sort_values("_sort", ascending=False).drop(columns="_sort")
    degenerate = {nice("prior"), nice("majority")}
    for c in ("Micro-F1 [95% CI]", "Macro-F1 [95% CI]", "Macro-AUC [95% CI]"):
        t3 = bold_best(t3, c, exclude_rows=degenerate)
    emit(t3, "T3", "Classification results",
         "Multi-label classification performance, 5-fold cross-validation "
         "(n=259 studies, 13 findings).",
         "Intervals are 95\\% bootstrap percentile intervals over 2{,}000 "
         "resamples of studies. Thresholds were tuned on training folds only. "
         "The class-prior and majority-label rows are non-learning baselines. "
         "Permutation p-values test against label-shuffled null "
         "(*p<0.05, **p<0.01, ***p<0.001).")
else:
    MISSING.append({"id": "T3", "title": "Classification results",
                    "reason": "results_with_ci.csv — run notebook 06"})

[ok] T3 — Classification results  (14x5)
                                     Model        Micro-F1 [95% CI]        Macro-F1 [95% CI]       Macro-AUC [95% CI]   p (perm.)
                     Multi-task VLM (ours) **0.835 [0.788, 0.878]** **0.797 [0.703, 0.853]** **0.945 [0.909, 0.975]** 9.99e-04***
              TF-IDF + logistic regression     0.671 [0.626, 0.718]     0.636 [0.563, 0.688]     0.923 [0.898, 0.947] 9.99e-04***
                Late fusion (image + text)     0.633 [0.588, 0.677]     0.604 [0.526, 0.660]     0.906 [0.875, 0.934] 9.99e-04***
                       TF-IDF + linear SVM     0.711 [0.663, 0.757]     0.584 [0.522, 0.633]     0.890 [0.856, 0.925] 9.99e-04***
                              ClinicalBERT     0.474 [0.432, 0.514]     0.450 [0.394, 0.491]     0.855 [0.816, 0.892] 9.99e-04***
        BiomedCLIP linear probe (findings)     0.340 [0.310, 0.371]     0.335 [0.288, 0.372]     0.798 [0.761, 0.830] 9.99e-04***
          EfficientNetV2-S + attention MIL     0.

## T4 — Report generation

Clinical efficacy first, surface metrics after. The column order encodes the
argument: a reader scanning left to right sees the clinically meaningful
numbers before BLEU.

In [6]:
gen = load_csv("generation_metrics.csv")
gci = load_csv("generation_ci.csv")

if gen is not None:
    ci_map = gci.set_index("method") if gci is not None else None
    rows = []
    for _, r in gen.iterrows():
        m = r["method"]
        if ci_map is not None and m in ci_map.index:
            ce = fmt_ci(r["CE_F1"], ci_map.loc[m, "CI_low"], ci_map.loc[m, "CI_high"])
        else:
            ce = fmt(r["CE_F1"])
        rows.append({
            "Method": nice(m),
            "CE-F1 [95% CI]": ce,
            "CE-P": fmt(r.get("CE_precision")),
            "CE-R": fmt(r.get("CE_recall")),
            "Exact set": fmt(r.get("CE_exact_set")),
            "BLEU-1": fmt(r.get("BLEU-1")),
            "BLEU-4": fmt(r.get("BLEU-4")),
            "ROUGE-L": fmt(r.get("ROUGE-L")),
            "METEOR": fmt(r.get("METEOR")),
            "Distinct": fmt(r.get("distinct"), 2),
        })
    t4 = pd.DataFrame(rows)
    t4["_s"] = [float(str(v).split(" ")[0]) for v in t4["CE-F1 [95% CI]"]]
    t4 = t4.sort_values("_s", ascending=False).drop(columns="_s")
    degen = {nice("constant_normal"), nice("random_train_report")}
    for c in ("CE-F1 [95% CI]", "ROUGE-L", "BLEU-4"):
        t4 = bold_best(t4, c, exclude_rows=degen)
    emit(t4, "T4", "Report generation",
         "Impression generation performance against the radiologist-written "
         "reference.",
         "CE = clinical efficacy: findings extracted from the generated "
         "impression using the ontology of Table S1, compared with those of "
         "the reference. Distinct = fraction of unique outputs. "
         "The constant-\"normal\" row is a degenerate baseline that produces "
         "one fixed sentence for every study; it is included to show what "
         "surface metrics reward.")

    cn = gen[gen["method"] == "constant_normal"]
    best_ce = gen.loc[gen["CE_F1"].idxmax()]
    if len(cn):
        c = cn.iloc[0]
        share = c["ROUGE-L"] / best_ce["ROUGE-L"] if best_ce["ROUGE-L"] else np.nan
        print(f">>> Sentence for the abstract and introduction:\n"
              f"    A degenerate generator that writes one fixed sentence for "
              f"every study reaches ROUGE-L {c['ROUGE-L']:.3f} "
              f"({share:.0%} of the best system's {best_ce['ROUGE-L']:.3f}) "
              f"while scoring only {c['CE_F1']:.3f} clinical-efficacy F1 and "
              f"naming the correct finding set in {c['CE_exact_set']:.0%} of "
              f"studies.\n")
        print("    Note on BLEU: the fixed sentence is shorter than the mean "
              "reference, so the\n    brevity penalty suppresses its BLEU "
              "scores. ROUGE-L is the fairer comparison\n    here and the one "
              "to quote — do not claim the baseline 'wins' on BLEU.")
else:
    MISSING.append({"id": "T4", "title": "Report generation",
                    "reason": "generation_metrics.csv — run notebook 05"})

[ok] T4 — Report generation  (5x10)
                        Method           CE-F1 [95% CI]  CE-P  CE-R Exact set BLEU-1    BLEU-4   ROUGE-L METEOR Distinct
       Consensus fusion (ours) **0.411 [0.358, 0.472]** 0.437 0.387     0.374  0.161     0.100     0.228  0.189     0.08
Constant "normal" (degenerate)     0.317 [0.264, 0.374] 0.351 0.289     0.324  0.000     0.000     0.159  0.118     0.00
       Retrieve-and-copy (K=1)     0.281 [0.230, 0.326] 0.274 0.289     0.251  0.279 **0.194** **0.234**  0.231     0.38
            Multi-task decoder     0.219 [0.185, 0.253] 0.158 0.356     0.085  0.094     0.037     0.140  0.224     0.96
        Random training report     0.153 [0.113, 0.193] 0.151 0.156     0.131  0.117     0.046     0.119  0.095     0.45

>>> Sentence for the abstract and introduction:
    A degenerate generator that writes one fixed sentence for every study reaches ROUGE-L 0.159 (70% of the best system's 0.228) while scoring only 0.317 clinical-efficacy F1 and naming the

## T5 — Retrieval quality

In [7]:
ret = load_csv("retrieval_metrics.csv")
if ret is not None:
    ks = sorted(int(c.split("@")[1]) for c in ret.columns if c.startswith("Recall@"))
    rows = []
    for _, r in ret.iterrows():
        row = {"Method": str(r["method"])}
        for k in ks:
            row[f"R@{k}"] = fmt(r.get(f"Recall@{k}"))
        for k in ks:
            row[f"P@{k}"] = fmt(r.get(f"Precision@{k}"))
        row["MRR"] = fmt(r.get("MRR"))
        rows.append(row)
    t5 = pd.DataFrame(rows)
    for c in t5.columns[1:]:
        t5 = bold_best(t5, c, exclude_rows={"random baseline"})
    emit(t5, "T5", "Retrieval quality",
         "Case retrieval performance over BiomedCLIP image embeddings.",
         "A retrieved study counts as relevant if it shares at least one "
         "finding with the query. The index excludes the query's own "
         "cross-validation fold and its near-duplicate group. "
         "R@K = Recall@K, P@K = Precision@K, MRR = mean reciprocal rank.")
else:
    MISSING.append({"id": "T5", "title": "Retrieval quality",
                    "reason": "retrieval_metrics.csv — run notebook 05"})

[ok] T5 — Retrieval quality  (2x10)
                    Method       R@1       R@3       R@5      R@10       P@1       P@3       P@5      P@10       MRR
BiomedCLIP image embedding **0.368** **0.628** **0.744** **0.830** **0.368** **0.344** **0.326** **0.296** **0.520**
           random baseline     0.202     0.466     0.614     0.785     0.202     0.200     0.197     0.193     0.370



## T6 — Ablation

Two panels in one table: component removal for classification, and the K
sweep for retrieval-augmented generation. Presenting them together keeps the
ablation section to a single float.

In [8]:
abl = load_csv("ablation_classification.csv")
sweep = load_csv("rag_k_sweep.csv")
panels = []

if abl is not None:
    for _, r in abl.iterrows():
        panels.append({
            "Panel": "A. Classification components",
            "Configuration": str(r["configuration"]).capitalize(),
            "Metric": "Macro-F1",
            "Value [95% CI]": fmt_ci(r["macro_F1"], r.get("CI_low"), r.get("CI_high")),
            "Δ vs full": (f"{r['delta_vs_full']:+.3f}"
                          if "delta_vs_full" in r and np.isfinite(r["delta_vs_full"])
                          else "—"),
        })

if sweep is not None:
    best_k = sweep.loc[sweep["CE_F1"].idxmax(), "K"]
    base_ce = sweep["CE_F1"].max()
    for _, r in sweep.iterrows():
        panels.append({
            "Panel": "B. Retrieved cases (K)",
            "Configuration": f"K = {int(r['K'])}" + ("  (selected)" if r["K"] == best_k else ""),
            "Metric": "CE-F1",
            "Value [95% CI]": fmt(r["CE_F1"]),
            "Δ vs full": f"{r['CE_F1'] - base_ce:+.3f}",
        })

if panels:
    t6 = pd.DataFrame(panels)
    emit(t6, "T6", "Ablation study",
         "Ablation of system components and of the number of retrieved cases.",
         "Panel A removes one component at a time from the full system; "
         "intervals are 95\\% bootstrap percentile intervals. Panel B varies K "
         "in retrieval-augmented consensus fusion. A configuration whose "
         "interval overlaps that of the full system is not shown to contribute "
         "at this sample size.")
else:
    MISSING.append({"id": "T6", "title": "Ablation study",
                    "reason": "ablation_classification.csv / rag_k_sweep.csv — notebook 06"})

[ok] T6 — Ablation study  (12x5)
                       Panel                   Configuration   Metric       Value [95% CI] Δ vs full
A. Classification components                     Full system Macro-F1 0.604 [0.526, 0.660]    +0.000
A. Classification components       Without text (image only) Macro-F1 0.219 [0.180, 0.253]    -0.386
A. Classification components       Without image (text only) Macro-F1 0.636 [0.563, 0.688]    +0.032
A. Classification components Without multi-task (probe only) Macro-F1 0.249 [0.207, 0.284]    -0.355
A. Classification components Without fine-tuning (zero-shot) Macro-F1 0.160 [0.137, 0.182]    -0.445
A. Classification components        Without learning (prior) Macro-F1 0.111 [0.103, 0.119]    -0.493
      B. Retrieved cases (K)                           K = 1    CE-F1                0.293    -0.118
      B. Retrieved cases (K)                           K = 2    CE-F1                0.324    -0.087
      B. Retrieved cases (K)                           K =

## T7 — Statistical comparisons

Study-level McNemar with Holm correction. Only the comparisons you actually
discuss belong in the main text — the full matrix goes to supplementary.

In [9]:
mc = load_csv("mcnemar_tests.csv")
if mc is not None and len(mc):
    m = mc.copy()
    m["_p"] = pd.to_numeric(m["p_value"], errors="coerce")
    m = m.sort_values("_p")
    keep = m.head(10)
    rows = []
    for _, r in keep.iterrows():
        rows.append({
            "Model A": nice(r["model_A"]),
            "Model B": nice(r["model_B"]),
            "A✓B✗": int(r["A_right_B_wrong"]),
            "B✓A✗": int(r["B_right_A_wrong"]),
            "p": fmt_p(r["p_value"]),
            "p (Holm)": fmt_p(r.get("p_holm")) + stars(r.get("p_holm")),
            "Favours": nice(r["better"]) if r["better"] != "tie" else "tie",
        })
    t7 = pd.DataFrame(rows)
    emit(t7, "T7", "Statistical comparisons",
         "Pairwise McNemar tests between models.",
         "Exact binomial McNemar test on study-level outcomes (a study counts "
         "as correct only if its entire predicted finding set matches), giving "
         "259 independent units. Holm--Bonferroni correction across all "
         f"{len(mc)} pairwise comparisons. "
         "The two count columns give studies where one model is correct and "
         "the other is not. "
         "(*p<0.05, **p<0.01, ***p<0.001).")
    print(f">>> Full matrix of {len(mc)} comparisons goes to supplementary S8.")
else:
    MISSING.append({"id": "T7", "title": "Statistical comparisons",
                    "reason": "mcnemar_tests.csv — run notebook 06"})

[ok] T7 — Statistical comparisons  (10x7)
                     Model A                                    Model B  A✓B✗  B✓A✗       p   p (Holm)                      Favours
       Multi-task VLM (ours)            BiomedCLIP linear probe (image)   206     0 <0.0001 <0.0001***        Multi-task VLM (ours)
        Class-prior baseline                      Multi-task VLM (ours)     0   206 <0.0001 <0.0001***        Multi-task VLM (ours)
    ResNet50 + attention MIL                      Multi-task VLM (ours)     1   199 <0.0001 <0.0001***        Multi-task VLM (ours)
       Multi-task VLM (ours)                       BiomedCLIP zero-shot   194     3 <0.0001 <0.0001***        Multi-task VLM (ours)
       Multi-task VLM (ours) BiomedCLIP linear probe (image + findings)   168     0 <0.0001 <0.0001***        Multi-task VLM (ours)
       Multi-task VLM (ours)         BiomedCLIP linear probe (findings)   175     2 <0.0001 <0.0001***        Multi-task VLM (ours)
 DenseNet121 + attention MIL      

---
# Supplementary tables

## S1 — Label ontology

Publish this. Without the exact rules your label extraction is not
reproducible, and "we used keyword matching" is not a method description.

In [10]:
onto = load_json("ontology.json")
if onto:
    rows = []
    for lab, spec in onto.items():
        inc = spec.get("include", []) if isinstance(spec, dict) else []
        exc = spec.get("exclude", []) if isinstance(spec, dict) else []
        grp = spec.get("group", "") if isinstance(spec, dict) else ""
        n_pos = ""
        if Ydf is not None and lab in Ydf.columns:
            n_pos = int(Ydf[lab].sum())
        tier = next((t for t, names in tiers.items()
                     if isinstance(names, list) and lab in names), "")
        rows.append({"Finding": lab, "Anatomical group": grp, "n": n_pos,
                     "Tier": tier,
                     "Inclusion patterns": "; ".join(inc),
                     "Exclusion patterns": "; ".join(exc) or "—"})
    s1 = pd.DataFrame(rows).sort_values(
        "n", ascending=False, key=lambda c: pd.to_numeric(c, errors="coerce"))
    emit(s1, "S1", "Label ontology",
         "Rule-based label ontology used to extract findings from the "
         "Impression section.",
         "Patterns are Python regular expressions applied case-insensitively. "
         "A finding is assigned when any inclusion pattern matches and no "
         "exclusion pattern matches.", show=False)
    print(s1[["Finding", "Anatomical group", "n", "Tier"]].head(15).to_string(index=False))
else:
    MISSING.append({"id": "S1", "title": "Label ontology",
                    "reason": "ontology.json — run notebook 00"})

[ok] S1 — Label ontology  (34x6)
                Finding Anatomical group  n      Tier
           Normal study           Normal 84 trainable
       Normal pregnancy        Obstetric 42 trainable
            Fatty liver    Hepatobiliary 28 trainable
               Cystitis          Urinary 21 trainable
           Bulky uterus          Uterine 16 trainable
           Hepatomegaly    Hepatobiliary 15 trainable
Early pregnancy failure        Obstetric 14 trainable
        Significant PVR          Urinary 10 trainable
      Pelvic collection           Pelvic  7  marginal
  Thickened endometrium          Uterine  6  marginal
             Renal cyst          Urinary  6  marginal
         Cholelithiasis    Hepatobiliary  5  marginal
           Ovarian cyst          Ovarian  5  marginal
      Prominent ovaries          Ovarian  4      rare
     Polycystic ovaries          Ovarian  4      rare


## S2 — Per-class performance

In [11]:
pc = load_csv("baseline_per_class.csv")
if pc is not None:
    best = None
    if ci is not None:
        real = ci[~ci["model"].isin(["prior", "majority"])]
        if len(real):
            best = real.sort_values("macro_F1", ascending=False).iloc[0]["model"]
    sub = pc[pc["model"] == best] if best else pc
    rows = [{"Finding": r["label"], "n": int(r["n_positive"]),
             "Precision": fmt(r["precision"]), "Recall": fmt(r["recall"]),
             "F1": fmt(r["f1"]), "AUC": fmt(r["auc"])}
            for _, r in sub.sort_values("n_positive", ascending=False).iterrows()]
    s2 = pd.DataFrame(rows)
    emit(s2, "S2", "Per-class performance",
         f"Per-finding classification performance for {nice(best)}.",
         "n = positive studies out of 259. Thresholds tuned on training folds "
         "only.")
    # All models, long form, for the record.
    pc.to_csv(TBL / "S2b_per_class_all_models.csv", index=False)
    REGISTRY.append({"id": "S2b", "title": "Per-class, all models",
                     "rows": len(pc), "cols": pc.shape[1],
                     "file": "S2b_per_class_all_models.csv",
                     "caption": "Per-class metrics for every model (CSV only)."})
else:
    MISSING.append({"id": "S2", "title": "Per-class performance",
                    "reason": "baseline_per_class.csv — run notebook 03"})

[missing] S2 — Per-class performance


## S3–S5 — Zero-shot, separability, attribution

In [12]:
zs = load_csv("zeroshot_auc.csv")
if zs is not None:
    s3 = pd.DataFrame({"Finding": zs["label"],
                       "Zero-shot AUC": [fmt(v) for v in zs["zeroshot_AUC"]]})
    emit(s3, "S3", "Zero-shot performance",
         "BiomedCLIP zero-shot classification AUC per finding, with no "
         "training on our corpus.",
         "Prompts were ensembled over 2--3 paraphrases per finding; 0.5 is "
         "chance.", show=False)
else:
    MISSING.append({"id": "S3", "title": "Zero-shot performance",
                    "reason": "zeroshot_auc.csv — run notebook 04"})

sep = load_csv("embedding_separability.csv")
if sep is not None:
    s4 = pd.DataFrame({"Finding": sep["label"], "n": sep["n"],
                       "Within-class sim.": [fmt(v, 4) for v in sep["within"]],
                       "Between-class sim.": [fmt(v, 4) for v in sep["between"]],
                       "Gap": [f"{v:+.4f}" for v in sep["gap"]]})
    emit(s4, "S4", "Embedding separability",
         "Cosine similarity within versus between finding classes in the "
         "frozen BiomedCLIP image embedding space.",
         "A positive gap indicates the pretrained space already separates that "
         "finding before any fine-tuning.", show=False)
else:
    MISSING.append({"id": "S4", "title": "Embedding separability",
                    "reason": "embedding_separability.csv — run notebook 04"})

attr = load_csv("text_attribution.csv")
if attr is not None:
    s5 = pd.DataFrame({"Finding": attr["label"],
                       "Highest-weighted terms": [
                           ", ".join(str(t).split("|")[:8]) for t in attr["top_terms"]]})
    emit(s5, "S5", "Term attribution",
         "Terms with the highest positive weight per finding in the linear "
         "text classifier.",
         "Terms are ordered by coefficient magnitude. Attribution is exact for "
         "a linear model: a token's contribution is its TF-IDF value times its "
         "coefficient.", show=False)
else:
    MISSING.append({"id": "S5", "title": "Term attribution",
                    "reason": "text_attribution.csv — run notebook 06"})

[ok] S3 — Zero-shot performance  (13x2)
[ok] S4 — Embedding separability  (13x5)
[ok] S5 — Term attribution  (13x2)


## S6 — Fold composition

Evidence that the stratification worked. One reviewer in three asks for this
when a paper reports cross-validation on an imbalanced corpus.

In [13]:
if df is not None and Ydf is not None and "fold" in df.columns:
    targets = [t for t in (tiers.get("trainable", []) + tiers.get("marginal", []))
               if t in Ydf.columns]
    rows = []
    for f in sorted(df["fold"].unique()):
        mask = df["fold"] == f
        row = {"Fold": int(f), "Studies": int(mask.sum())}
        for t in targets:
            row[t] = int(Ydf.loc[mask.values, t].sum())
        rows.append(row)
    tot = {"Fold": "Total", "Studies": len(df)}
    for t in targets:
        tot[t] = int(Ydf[t].sum())
    rows.append(tot)
    s6 = pd.DataFrame(rows)
    emit(s6, "S6", "Fold composition",
         "Distribution of findings across cross-validation folds.",
         "Folds were generated with group-stratified iterative stratification: "
         "near-duplicate studies are kept together and label proportions are "
         "balanced across folds.", show=False)
    print(s6.to_string(index=False))

[ok] S6 — Fold composition  (6x15)
 Fold  Studies  Normal study  Normal pregnancy  Fatty liver  Cystitis  Bulky uterus  Hepatomegaly  Early pregnancy failure  Significant PVR  Pelvic collection  Renal cyst  Thickened endometrium  Cholelithiasis  Ovarian cyst
    0       52            17                 9            6         4             3             3                        3                2                  2           1                      2               1             1
    1       52            16                 9            5         4             3             3                        3                2                  1           1                      1               1             1
    2       52            17                 8            5         4             4             3                        3                2                  1           1                      1               1             1
    3       52            17                 8            6         4

## S7 — Generated report examples

A handful of verbatim outputs beside their references. Include successes and
failures; a supplementary table of only successes reads as cherry-picking.

In [14]:
gr = load_csv("generated_reports.csv")
cases = load_csv("rag_case_studies.csv")
if cases is not None and len(cases):
    s7 = pd.DataFrame({
        "Study": cases["study_id"],
        "Outcome": cases.get("outcome", ""),
        "Reference impression": cases["reference"].astype(str).str.slice(0, 160),
        "Generated impression": cases["generated"].astype(str).str.slice(0, 160),
        "Reference findings": cases.get("ref_labels", ""),
        "Extracted findings": cases.get("gen_labels", ""),
    })
    emit(s7, "S7", "Generated report examples",
         "Representative generated impressions with their references.",
         "Findings columns show the output of the ontology of Table S1 applied "
         "to each text.", show=False)
    print(s7[["Study", "Outcome"]].to_string(index=False))
elif gr is not None:
    meth = [c for c in gr.columns if c not in ("study_id", "reference")]
    pick = "consensus_fusion@K" if "consensus_fusion@K" in meth else meth[0]
    s7 = pd.DataFrame({
        "Study": gr["study_id"].head(8),
        "Reference impression": gr["reference"].astype(str).str.slice(0, 160).head(8),
        "Generated impression": gr[pick].astype(str).str.slice(0, 160).head(8)})
    emit(s7, "S7", "Generated report examples",
         "Representative generated impressions with their references.",
         show=False)
else:
    MISSING.append({"id": "S7", "title": "Generated report examples",
                    "reason": "rag_case_studies.csv — run notebook 05"})

[ok] S7 — Generated report examples  (4x6)
Study   Outcome
   R4   correct
   R6   correct
   R1 incorrect
   R2 incorrect


## S8 — Full statistical comparison matrix

In [15]:
if mc is not None and len(mc):
    s8 = pd.DataFrame({
        "Model A": [nice(v) for v in mc["model_A"]],
        "Model B": [nice(v) for v in mc["model_B"]],
        "A✓B✗": mc["A_right_B_wrong"],
        "B✓A✗": mc["B_right_A_wrong"],
        "p (study-level)": [fmt_p(v) for v in mc["p_value"]],
        "p (label-level)": [fmt_p(v) for v in mc.get("p_value_label_level", mc["p_value"])],
        "p (Holm)": [fmt_p(v) for v in mc.get("p_holm", mc["p_value"])],
        "Significant": ["yes" if v else "no"
                        for v in mc.get("significant_holm", mc["significant_0.05"])],
    })
    emit(s8, "S8", "Full statistical comparison matrix",
         "All pairwise McNemar comparisons between models.",
         "Study-level tests treat each study as one unit (correct only if the "
         "full finding set matches). Label-level tests treat each "
         "(study, finding) decision as independent, which overstates "
         "significance because findings within a study are correlated; they "
         "are reported for completeness only.", show=False)

[ok] S8 — Full statistical comparison matrix  (91x8)


## S9 — Label audit

If you have completed the manual audit (gap 1 in the paper plan), this table
reports the agreement rate. It is the single cheapest thing you can add to
strengthen the manuscript.

In [16]:
audit = load_csv("label_audit_reviewed.csv") or load_csv("label_audit.csv")
if audit is not None:
    reviewed = [c for c in audit.columns
                if c.lower() in ("corrected_labels", "reviewer_labels",
                                 "corrected", "reviewed_labels")]
    if reviewed:
        col = reviewed[0]
        orig = audit.get("labels", audit.get("extracted_labels"))
        done = audit[col].notna() & (audit[col].astype(str).str.strip() != "")
        agree = (audit.loc[done, col].astype(str).str.strip() ==
                 orig.loc[done].astype(str).str.strip())
        s9 = pd.DataFrame([
            {"Quantity": "Studies audited", "Value": f"{int(done.sum())} / {len(audit)}"},
            {"Quantity": "Exact agreement with rule extraction",
             "Value": f"{agree.sum()} ({agree.mean():.1%})"},
            {"Quantity": "Studies corrected",
             "Value": f"{int((~agree).sum())} ({(~agree).mean():.1%})"},
            {"Quantity": "Cohen's κ (rule vs reviewer)", "Value": "TODO — compute per finding"},
            {"Quantity": "Second reviewer subset", "Value": "TODO"},
            {"Quantity": "Inter-rater agreement", "Value": "TODO"},
        ])
        emit(s9, "S9", "Label audit",
             "Agreement between rule-based label extraction and manual review.",
             "Manual review was performed by a clinician against the "
             "Impression text alone, blinded to model output.")
    else:
        print("[pending] S9 — label_audit.csv found but no reviewer column yet.\n"
              "          Add a 'corrected_labels' column, fill it in, and re-run.\n"
              "          This is the highest-value single addition to the manuscript.")
        MISSING.append({"id": "S9", "title": "Label audit",
                        "reason": "audit not yet completed by a clinician"})
else:
    MISSING.append({"id": "S9", "title": "Label audit",
                    "reason": "label_audit.csv — run notebook 00, then review it"})

[pending] S9 — label_audit.csv found but no reviewer column yet.
          Add a 'corrected_labels' column, fill it in, and re-run.
          This is the highest-value single addition to the manuscript.


---
# Completeness audit

What exists, what does not, and which notebook produces each missing piece.

In [17]:
reg = pd.DataFrame(REGISTRY)
if len(reg):
    reg = reg[["id", "title", "rows", "cols", "file"]]
    print("BUILT")
    print(reg.to_string(index=False))
    reg.to_csv(TBL / "table_manifest.csv", index=False)

if MISSING:
    miss = pd.DataFrame(MISSING)
    print("\nNOT BUILT")
    print(miss.to_string(index=False))
    miss.to_csv(TBL / "table_gaps.csv", index=False)
else:
    print("\nAll tables built.")

main = [r["id"] for r in REGISTRY if r["id"].startswith("T")]
supp = [r["id"] for r in REGISTRY if r["id"].startswith("S")]
print(f"\nMain text: {len(main)}/7 ({', '.join(sorted(main)) or 'none'})")
print(f"Supplementary: {len(supp)}/9 ({', '.join(sorted(supp)) or 'none'})")

BUILT
 id                              title  rows  cols                                  file
 T1            Dataset characteristics    20     2            T1_dataset_characteristics
 T2             Implementation details    25     2             T2_implementation_details
 T3             Classification results    14     5             T3_classification_results
 T4                  Report generation     5    10                  T4_report_generation
 T5                  Retrieval quality     2    10                  T5_retrieval_quality
 T6                     Ablation study    12     5                     T6_ablation_study
 T7            Statistical comparisons    10     7            T7_statistical_comparisons
 S1                     Label ontology    34     6                     S1_label_ontology
S2b              Per-class, all models   117     7          S2b_per_class_all_models.csv
 S3              Zero-shot performance    13     2              S3_zero_shot_performance
 S4            

## Manuscript-readiness checks

Four properties a reviewer will verify. Each maps to a checklist line in the
paper plan.

In [18]:
checks = []


def check(name, ok, detail):
    checks.append({"Check": name,
                   "Status": "PASS" if ok else "ACTION NEEDED",
                   "Detail": detail})


if ci is not None:
    has_base = {"prior", "majority"} <= set(ci["model"])
    check("Non-learning baselines in T3", has_base,
          "class-prior and majority rows present" if has_base
          else "add them — reviewers compute the floor themselves")
    has_ci = ci["macro_F1_95CI"].notna().all()
    check("Every classification result has a CI", has_ci,
          "all rows carry bootstrap intervals" if has_ci else "some rows lack intervals")

if gen is not None:
    has_cn = "constant_normal" in set(gen["method"])
    check("Degenerate baseline in T4", has_cn,
          "constant-normal row present — quote it in the abstract" if has_cn
          else "add it; it is the strongest argument you have")

if onto:
    check("Ontology publishable as S1", True,
          f"{len(onto)} findings with explicit regex rules")

check("Label audit completed", any(r["id"] == "S9" for r in REGISTRY),
      "agreement rate available" if any(r["id"] == "S9" for r in REGISTRY)
      else "not done — highest-value remaining addition (1-2 days clinician time)")

todo_rows = int(t2["Value"].astype(str).str.contains("TODO").sum())
check("T2 fully specified", todo_rows == 0,
      "all settings recorded" if todo_rows == 0
      else f"{todo_rows} settings still marked TODO (hardware, versions)")

chk = pd.DataFrame(checks)
chk.to_csv(TBL / "readiness_checks.csv", index=False)
print(chk.to_string(index=False))

                               Check        Status                                                                Detail
        Non-learning baselines in T3          PASS                                 class-prior and majority rows present
Every classification result has a CI          PASS                                    all rows carry bootstrap intervals
           Degenerate baseline in T4          PASS                constant-normal row present — quote it in the abstract
          Ontology publishable as S1          PASS                                 34 findings with explicit regex rules
               Label audit completed ACTION NEEDED not done — highest-value remaining addition (1-2 days clinician time)
                  T2 fully specified ACTION NEEDED                     3 settings still marked TODO (hardware, versions)


## Package everything

In [19]:
zip_path = Path("/kaggle/working/manuscript_tables.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(TBL.glob("*")):
        if p.is_file():
            sub = ("csv" if p.suffix == ".csv" else
                   "latex" if p.suffix == ".tex" else
                   "markdown" if p.suffix == ".md" else "other")
            zf.write(p, arcname=f"{sub}/{p.name}")

n_files = len(list(TBL.glob("*")))
print(f"{zip_path} — {zip_path.stat().st_size/1024:.0f} KB, {n_files} files")
print("\nLaTeX usage:")
print(r"  \input{tables/T3_classification_results.tex}")
print("\nWord/Docs: open the .md file, copy the table block, paste, then")
print("Insert > Table > Convert text to table (delimiter: |).")

/kaggle/working/manuscript_tables.zip — 38 KB, 46 files

LaTeX usage:
  \input{tables/T3_classification_results.tex}

Word/Docs: open the .md file, copy the table block, paste, then
Insert > Table > Convert text to table (delimiter: |).


In [20]:
combined = TBL / "ALL_TABLES.md"
parts = ["# Manuscript tables\n",
         "Generated from the analysis artifacts. Main text first, then "
         "supplementary.\n"]
for tid in sorted([r["id"] for r in REGISTRY],
                  key=lambda s: (s[0] != "T", s)):
    rec = next(r for r in REGISTRY if r["id"] == tid)
    md = TBL / f"{rec['file']}.md"
    if md.exists():
        parts.append(md.read_text())
        parts.append("\n\n---\n")
combined.write_text("\n".join(parts))
print(f"Single-document version: {combined}")

Single-document version: /kaggle/working/artifacts/tables/ALL_TABLES.md


## Using these

**Numbers are locked here.** Once you paste T3 and T4 into the manuscript,
do not retype a number anywhere else in the text — read it off the table. The
most common arithmetic error in student papers is an abstract that quotes a
figure the results section contradicts, and it is the kind of thing a
reviewer notices immediately and generalises from.

**Two items need your input before submission:**

1. **T2** has TODO rows for hardware and framework versions. Fill them in
   from your Kaggle session.
2. **S9** needs the clinician audit. Add a `corrected_labels` column to
   `label_audit.csv`, have it reviewed, save it as
   `label_audit_reviewed.csv`, and re-run this notebook — the agreement rate
   then appears automatically.

**One sentence worth building the abstract around,** printed above when T4
builds: the degenerate baseline's ROUGE-L against its clinical-efficacy F1.
Stating that a generator which writes one fixed sentence scores competitively
on surface metrics is the clearest possible motivation for your evaluation
protocol, and very few papers in this area report it.